# Advanced RAG — Reranking Deep Dive (2026 Edition)
**IT AI Enthusiast — Mayank Chugh | @itaienthusiast**

> **GitHub:** github.com/mayankchugh-learning

| | Model/API and Version |
|---|---|
| **Cohere client** | `cohere.ClientV2()` ✅ |
| **Cohere model** | `rerank-v3.5` / `rerank-v4.0-pro` ✅ |
| **BM25 library** | `bm25s` (up to 500× faster) ✅ |
| **Embedding model** | `all-MiniLM-L6-v2` / `BAAI/bge-m3` ✅ |
| **sentence-transformers** | v5.x (`encode_query`, `CrossEncoder.rank`) ✅ |
| **LangChain imports** | `langchain_cohere`, `langchain_classic` ✅ |

In [1]:
# ── Install all dependencies ──────────────────────────────────────────────
# Core
%pip install -q sentence-transformers rank_bm25 bm25s scikit-learn numpy

# Cohere SDK v5+ (required for ClientV2 and langchain-cohere)
%pip install -q "cohere>=5.0"

# 2026 frontier open-source rerankers
%pip install -q FlagEmbedding mxbai_rerank voyageai

# LangChain v1.0 (Oct 2025 GA) + integration packages
%pip install -q langchain langchain-cohere langchain-community langchain-classic

print("All packages installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sourc

In [2]:
# ── Verify key versions ────────────────────────────────────────────────────
import importlib
pkgs = ["sentence_transformers","cohere","rank_bm25","bm25s","langchain","langchain_cohere","numpy","sklearn"]
for p in pkgs:
    try:
        m = importlib.import_module(p)
        print(f"  {p:<28} {getattr(m,'__version__','n/a')}")
    except ImportError:
        print(f"  {p:<28} NOT INSTALLED")

  sentence_transformers        5.5.1
  cohere                       5.21.1
  rank_bm25                    n/a
  bm25s                        0.3.9
  langchain                    1.3.9
  langchain_cohere             n/a
  numpy                        2.0.2
  sklearn                      1.6.1


## Methods Overview — Before We Code

In [3]:
documents = [
    "A collection of articles is stored for retrieval purposes.",
    "Indexing relies heavily on how terms are weighted across a corpus.",
    "Parsing text helps isolate the key terms in a passage.",
    "Term weighting strategies often depend on sparse vector representations.",
    "Knowing how a paragraph is structured helps when isolating key terms.",
    "A well-tuned term extractor improves the precision of a search engine.",
    "Vector closeness can reveal how well two passages relate in meaning.",
    "Statistical models help automate the process of identifying key terms."
]
query = "Modern language models boost the precision of automated term extraction."

print(f"Corpus size : {len(documents)} passages")
print(f"Query       : {query}")

Corpus size : 8 passages
Query       : Modern language models boost the precision of automated term extraction.


## Stage 1 — Dense Retrieval (Bi-Encoder)

In [4]:
import warnings, os
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2 — fast general-purpose English embedding model
# For multilingual: SentenceTransformer("BAAI/bge-m3")  # 100+ langs, 8192 ctx
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print(f"Model          : all-MiniLM-L6-v2")
print(f"Embedding dim  : {model.get_sentence_embedding_dimension()}")
print(f"Max seq length : {model.max_seq_length} tokens")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model          : all-MiniLM-L6-v2
Embedding dim  : 384
Max seq length : 256 tokens


In [5]:
doc_embeddings = model.encode(documents, show_progress_bar=False)
print(f"Passage embeddings shape: {doc_embeddings.shape}")

Passage embeddings shape: (8, 384)


In [6]:
query_embedding = model.encode([query], show_progress_bar=False)

# Classic approach (still valid in v5)
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
sims_sklearn = sk_cosine(query_embedding, doc_embeddings)[0]

# v5 idiomatic approach (preferred)
sims = model.similarity(query_embedding, doc_embeddings)[0].numpy()

print("Both approaches match:", np.allclose(sims_sklearn, sims, atol=1e-5))
print(f"\nSimilarity scores: {sims.round(4)}")

Both approaches match: True

Similarity scores: [0.2185 0.488  0.4492 0.482  0.4119 0.7084 0.2301 0.5874]


In [7]:
sorted_idx = np.argsort(sims)[::-1]

print("=== Stage 1 — Ranked Passages (Bi-Encoder) ===")
print(f"{'Rank':<6} {'Sim':>6}  Passage")
print("-" * 70)
for rank, idx in enumerate(sorted_idx, 1):
    print(f"  {rank:<4} {sims[idx]:>6.4f}  {documents[idx]}")

=== Stage 1 — Ranked Passages (Bi-Encoder) ===
Rank      Sim  Passage
----------------------------------------------------------------------
  1    0.7084  A well-tuned term extractor improves the precision of a search engine.
  2    0.5874  Statistical models help automate the process of identifying key terms.
  3    0.4880  Indexing relies heavily on how terms are weighted across a corpus.
  4    0.4820  Term weighting strategies often depend on sparse vector representations.
  5    0.4492  Parsing text helps isolate the key terms in a passage.
  6    0.4119  Knowing how a paragraph is structured helps when isolating key terms.
  7    0.2301  Vector closeness can reveal how well two passages relate in meaning.
  8    0.2185  A collection of articles is stored for retrieval purposes.


In [9]:
selected = [documents[i] for i in sorted_idx[:4]]

print("=== Top-4 candidates passed to all rerankers ===")
for i, d in enumerate(selected):
    print(f"  [{i}] {d}")

=== Top-4 candidates passed to all rerankers ===
  [0] A well-tuned term extractor improves the precision of a search engine.
  [1] Statistical models help automate the process of identifying key terms.
  [2] Indexing relies heavily on how terms are weighted across a corpus.
  [3] Term weighting strategies often depend on sparse vector representations.


## Stage 2a — BM25 Lexical Reranking

In [10]:
# ── A) rank_bm25 — classic approach ─────────────────────────────────────────
from rank_bm25 import BM25Okapi

tokenized_corpus = [d.lower().split() for d in selected]
tokenized_query  = query.lower().split()

bm25_engine  = BM25Okapi(tokenized_corpus)
bm25_results = bm25_engine.get_scores(tokenized_query)

print("=== rank_bm25 (classic) ===")
for rank, (idx, score) in enumerate(sorted(enumerate(bm25_results), key=lambda x: x[1], reverse=True), 1):
    print(f"  Rank {rank}: [{idx}] score={score:.4f} | {selected[idx]}")

=== rank_bm25 (classic) ===
  Rank 1: [1] score=0.8567 | Statistical models help automate the process of identifying key terms.
  Rank 2: [0] score=0.8203 | A well-tuned term extractor improves the precision of a search engine.
  Rank 3: [2] score=0.0000 | Indexing relies heavily on how terms are weighted across a corpus.
  Rank 4: [3] score=0.0000 | Term weighting strategies often depend on sparse vector representations.


In [11]:
# ── B) bm25s — 2026 recommended (up to 500x faster on large corpora) ───────
import bm25s

corpus_tok = bm25s.tokenize(selected, stopwords="en")
query_tok  = bm25s.tokenize([query],  stopwords="en")

retriever = bm25s.BM25()
retriever.index(corpus_tok)
hits, hit_scores = retriever.retrieve(query_tok, k=4)

print("=== bm25s (2026 recommended) ===")
for rank, (idx, score) in enumerate(zip(hits[0], hit_scores[0]), 1):
    print(f"  Rank {rank}: [{idx}] score={score:.4f} | {selected[idx]}")

# Speed context (from BM25S paper arXiv:2407.03618):
# bm25s vs rank_bm25 on 100k passages: ~1196 QPS vs ~225 QPS — up to 500x faster

Split strings:   0%|          | 0/4 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/4 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/4 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

=== bm25s (2026 recommended) ===
  Rank 1: [0] score=0.7588 | A well-tuned term extractor improves the precision of a search engine.
  Rank 2: [1] score=0.4816 | Statistical models help automate the process of identifying key terms.
  Rank 3: [3] score=0.2773 | Term weighting strategies often depend on sparse vector representations.
  Rank 4: [2] score=0.0000 | Indexing relies heavily on how terms are weighted across a corpus.


## Stage 2b — Cross-Encoder Reranking

In [12]:
import warnings, os
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from sentence_transformers import CrossEncoder

# Baseline model — still a valid fast choice for 2026
# Modern upgrade: CrossEncoder("Alibaba-NLP/gte-reranker-modernbert-base")  # 8192 ctx
ce_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("CrossEncoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

CrossEncoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [13]:
# ── Classic API: .predict() — still valid in v5 ─────────────────────────────
sentence_pairs = [[query, d] for d in selected]
raw_scores = ce_model.predict(sentence_pairs)

print("=== CrossEncoder.predict() — raw logit scores (unbounded, higher = more relevant) ===")
for i, (d, s) in enumerate(zip(selected, raw_scores)):
    print(f"  [{i}] score={s:+.4f} | {d}")

=== CrossEncoder.predict() — raw logit scores (unbounded, higher = more relevant) ===
  [0] score=+0.8530 | A well-tuned term extractor improves the precision of a search engine.
  [1] score=-7.4414 | Statistical models help automate the process of identifying key terms.
  [2] score=-10.9948 | Indexing relies heavily on how terms are weighted across a corpus.
  [3] score=-10.7988 | Term weighting strategies often depend on sparse vector representations.


In [14]:
# ── v5 new API: .rank() — returns sorted results directly ───────────────────
sorted_results = ce_model.rank(query, selected, return_documents=True)

print("=== CrossEncoder.rank() — v5 idiomatic API ===")
print(f"{'Rank':<6} {'Score':>8}  Passage")
print("-" * 70)
for r in sorted_results:
    print(f"  {r['corpus_id']+1:<4} {r['score']:>+8.4f}  {r['text']}")

=== CrossEncoder.rank() — v5 idiomatic API ===
Rank      Score  Passage
----------------------------------------------------------------------
  1     +0.8530  A well-tuned term extractor improves the precision of a search engine.
  2     -7.4414  Statistical models help automate the process of identifying key terms.
  4    -10.7988  Term weighting strategies often depend on sparse vector representations.
  3    -10.9948  Indexing relies heavily on how terms are weighted across a corpus.


In [15]:
# ── 2026 upgrade path — same API, just change the model string ──────────────
# CrossEncoder("Alibaba-NLP/gte-reranker-modernbert-base")  # 150M, 8192-token ctx
# CrossEncoder("mixedbread-ai/mxbai-rerank-base-v2")        # 500M, RL-trained, Apache 2.0
# Both use identical .predict() and .rank() API
print("Upgrade path: change model string only — .predict() and .rank() are identical.")
print("   Alibaba-NLP/gte-reranker-modernbert-base : 8192-token ctx")
print("   mixedbread-ai/mxbai-rerank-base-v2       : RL-trained, Apache 2.0")

Upgrade path: change model string only — .predict() and .rank() are identical.
   Alibaba-NLP/gte-reranker-modernbert-base : 8192-token ctx
   mixedbread-ai/mxbai-rerank-base-v2       : RL-trained, Apache 2.0


## Stage 2c — Cohere Rerank API (2026 Syntax)

| | Syntax - 2026 |
|---|---|
| Client | `cohere.ClientV2()` ✅ |
| Model | `rerank-v3.5` or `rerank-v4.0-pro` ✅ |
| Response | look up via `result.index` ✅ |

In [16]:
import os, cohere

# Try Colab secrets first, then environment variable
try:
    from google.colab import userdata
    COHERE_API_KEY = userdata.get('COHERE_API_KEY')
    print('COHERE_API_KEY loaded from Colab secrets')
except Exception:
    COHERE_API_KEY = os.getenv('COHERE_API_KEY', 'YOUR_API_KEY_HERE')
    print(f'COHERE_API_KEY from env: {COHERE_API_KEY[:8]}...' if COHERE_API_KEY != 'YOUR_API_KEY_HERE' else 'WARNING: Using placeholder key')

# 2026: ClientV2 — current cohere SDK
co = cohere.ClientV2(api_key=COHERE_API_KEY)
print('Cohere ClientV2 instantiated')
print("  Models: 'rerank-v3.5' | 'rerank-v4.0-fast' | 'rerank-v4.0-pro'")

COHERE_API_KEY loaded from Colab secrets
Cohere ClientV2 instantiated
  Models: 'rerank-v3.5' | 'rerank-v4.0-fast' | 'rerank-v4.0-pro'


In [17]:
# ── Call rerank API ──────────────────────────────────────────────────────────
# 2026 (cohere>=5.21): return_documents param removed — use result.index to look up text
try:
    response = co.rerank(
        model='rerank-v3.5',          # was: 'rerank-english-v3.0'
        query=query,
        documents=selected,
        top_n=4
        # return_documents removed in cohere>=5.21 — use selected[result.index] instead
    )
    print(f'Results returned: {len(response.results)}')
    cohere_available = True
except Exception as e:
    print(f'Cohere API error: {e}')
    print('NOTE: Set COHERE_API_KEY in Colab secrets to enable Cohere reranking')
    response = None
    cohere_available = False

Results returned: 4


In [18]:
# ── Parse v2 response ────────────────────────────────────────────────────────
# response.results  → list of V2RerankResponseResultsItem
# 2026 (cohere>=5.21): each item has .index and .relevance_score only
# Use selected[result.index] to get the passage text

if cohere_available and response:
    print('=== Cohere rerank-v3.5 Results ===')
    print(f"{'Rank':<6} {'Index':>5}  {'Relevance':>9}  Passage")
    print('-' * 75)
    for rank, result in enumerate(response.results, 1):
        text = selected[result.index]   # 2026: index lookup (no .document attr)
        print(f'  {rank:<4} {result.index:>5}  {result.relevance_score:>9.4f}  {text}')
else:
    print('Skipping Cohere results display — API key not available')
    print('Expected ranking shape (from a previous authenticated run):')
    print('  Rank 1: A well-tuned term extractor improves the precision of a search engine.')
    print('  Rank 2: Statistical models help automate the process of identifying key terms.')

=== Cohere rerank-v3.5 Results ===
Rank   Index  Relevance  Passage
---------------------------------------------------------------------------
  1        1     0.3660  Statistical models help automate the process of identifying key terms.
  2        0     0.3208  A well-tuned term extractor improves the precision of a search engine.
  3        2     0.1115  Indexing relies heavily on how terms are weighted across a corpus.
  4        3     0.0701  Term weighting strategies often depend on sparse vector representations.


In [19]:
# ── Context window reference ─────────────────────────────────────────────────
print("Cohere model context windows:")
print("  rerank-english-v3.0  : 4,096  tokens — LEGACY, English only")
print("  rerank-v3.5          : 4,096  tokens — multilingual 100+ langs")
print("  rerank-v4.0-fast     : 32,764 tokens — Dec 2025, low latency")
print("  rerank-v4.0-pro      : 32,764 tokens — Dec 2025, max quality")

Cohere model context windows:
  rerank-english-v3.0  : 4,096  tokens — LEGACY, English only
  rerank-v3.5          : 4,096  tokens — multilingual 100+ langs
  rerank-v4.0-fast     : 32,764 tokens — Dec 2025, low latency
  rerank-v4.0-pro      : 32,764 tokens — Dec 2025, max quality


## Section — The 2026 Reranker Frontier

In [20]:
# ── BGE Reranker v2-m3 (Apache 2.0, 568M, multilingual) ─────────────────────
# pip install FlagEmbedding
# from FlagEmbedding import FlagReranker
# reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)
# scores = reranker.compute_score([[query, d] for d in selected], normalize=True)
# For LLM-based: from FlagEmbedding import FlagLLMReranker (bge-reranker-v2-gemma)
print("BGE v2-m3 : Apache 2.0 | 568M | multilingual | FlagEmbedding library")
print("Usage     : reranker.compute_score(pairs, normalize=True)  -> [0,1] scores")

BGE v2-m3 : Apache 2.0 | 568M | multilingual | FlagEmbedding library
Usage     : reranker.compute_score(pairs, normalize=True)  -> [0,1] scores


In [21]:
# ── mxbai-rerank-v2 (Apache 2.0, RL-trained, instruction steering) ──────────
# pip install mxbai_rerank
# from mxbai_rerank import MxbaiRerankV2
# mxbai = MxbaiRerankV2("mixedbread-ai/mxbai-rerank-base-v2")
# results = mxbai.rank(query=query, documents=selected)
# With instruction: mxbai.rank(query, docs, instruction="Rank by technical accuracy")
# Also as CrossEncoder: CrossEncoder("mixedbread-ai/mxbai-rerank-large-v2")
print("mxbai-rerank-v2 : Apache 2.0 | RL-trained (GRPO+contrastive) | instruction steering")
print("Sizes           : base-v2 (500M) or large-v2 (1.5B)")

mxbai-rerank-v2 : Apache 2.0 | RL-trained (GRPO+contrastive) | instruction steering
Sizes           : base-v2 (500M) or large-v2 (1.5B)


In [22]:
# ── Jina Reranker v3 (SOTA BEIR 61.94, CC-BY-NC) ────────────────────────────
# CC-BY-NC 4.0 -- non-commercial only without paid Jina agreement
# Via API (no GPU):
# import requests
# resp = requests.post("https://api.jina.ai/v1/rerank",
#     headers={"Authorization": "Bearer JINA_KEY"},
#     json={"model": "jina-reranker-v3", "query": query, "documents": selected})
# Via HuggingFace (GPU + trust_remote_code):
# CrossEncoder("jinaai/jina-reranker-v2-base-multilingual", trust_remote_code=True)
print("jina-reranker-v3 : SOTA BEIR 61.94 nDCG@10 | 131k ctx | 0.6B | CC-BY-NC")
print("jina-reranker-v2 : 278M | 100+ langs | trust_remote_code=True")

jina-reranker-v3 : SOTA BEIR 61.94 nDCG@10 | 131k ctx | 0.6B | CC-BY-NC
jina-reranker-v2 : 278M | 100+ langs | trust_remote_code=True


In [23]:
# ── Qwen3-Reranker (Apache 2.0, best open family, vLLM-compatible) ──────────
# Sizes: 0.6B / 4B / 8B — Apache 2.0
# Via vLLM (Cohere-compatible /v1/rerank endpoint):
#   vllm serve Qwen/Qwen3-Reranker-0.6B --task rerank --port 8000
#   local_co = cohere.ClientV2(api_key="dummy", base_url="http://localhost:8000")
#   response = local_co.rerank(model="Qwen3-Reranker-0.6B", query=query, documents=docs)
# Scoring: log P(yes|q,d) / (P(yes|q,d) + P(no|q,d))  -- causal LM log-odds
print("Qwen3-Reranker : Apache 2.0 | 0.6B/4B/8B | vLLM /v1/rerank endpoint")
print("Scoring        : causal LM log-odds -- log P(yes)/(P(yes)+P(no))")

Qwen3-Reranker : Apache 2.0 | 0.6B/4B/8B | vLLM /v1/rerank endpoint
Scoring        : causal LM log-odds -- log P(yes)/(P(yes)+P(no))


In [24]:
# ── Voyage rerank-2.5 ($0.05/1M tokens, 32k ctx, instruction-following) ─────
# import voyageai
# vo = voyageai.Client()  # VOYAGE_API_KEY env var
# reranking = vo.rerank(query=query, documents=selected, model="rerank-2.5", top_k=4)
# for r in reranking.results: print(r.relevance_score, r.document)
# rerank-2.5-lite : $0.02/1M tokens (cost-optimised)
# First 200M tokens free on signup
print("Voyage rerank-2.5      : $0.05/1M tokens | 32k ctx | instruction-following")
print("Voyage rerank-2.5-lite : $0.02/1M tokens | cost-optimised | 200M free")

Voyage rerank-2.5      : $0.05/1M tokens | 32k ctx | instruction-following
Voyage rerank-2.5-lite : $0.02/1M tokens | cost-optimised | 200M free


## Section — LangChain v1.0 Integration (Oct 2025)

In [25]:
# ── LangChain v1.0 — correct import paths (2026) ─────────────────────────────
# OLD: from langchain.retrievers.document_compressors import CohereRerank
# NEW:
# from langchain_cohere import CohereRerank
# from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

# compressor = CohereRerank(model="rerank-v3.5")  # model name is MANDATORY in v1.0

# Open-source cross-encoder in LangChain (new in v1.0):
# from langchain.retrievers.document_compressors import CrossEncoderReranker
# from langchain_community.cross_encoders import HuggingFaceCrossEncoder
# hf_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
# compressor = CrossEncoderReranker(model=hf_model, top_n=3)

print("LangChain v1.0 import paths:")
print("  CohereRerank         -> from langchain_cohere import CohereRerank")
print("  CrossEncoderReranker -> from langchain.retrievers.document_compressors import CrossEncoderReranker")
print("  HuggingFaceCrossEnc  -> from langchain_community.cross_encoders import HuggingFaceCrossEncoder")
print("  ContextualCompr.     -> from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever")

LangChain v1.0 import paths:
  CohereRerank         -> from langchain_cohere import CohereRerank
  CrossEncoderReranker -> from langchain.retrievers.document_compressors import CrossEncoderReranker
  HuggingFaceCrossEnc  -> from langchain_community.cross_encoders import HuggingFaceCrossEncoder
  ContextualCompr.     -> from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever


## Section — Side-by-Side Comparison

In [26]:
print("=" * 78)
print(f"  QUERY: {query}")
print("=" * 78)
print()
print("Stage 1 -- Dense (all-MiniLM-L6-v2)")
for i,(s,d) in enumerate([(0.7084, 'A well-tuned term extractor improves the precision of a search engine.'), (0.5874, 'Statistical models help automate the process of identifying key terms.'), (0.488, 'Indexing relies heavily on how terms are weighted across a corpus.'), (0.482, 'Term weighting strategies often depend on sparse vector representations.')],1):
    print(f"  {i}. {s:.4f}  {d}")
print()
print("Stage 2a -- rank_bm25 (classic)")
for i,(s,d) in enumerate([(0.8567, 'Statistical models help automate the process of identifying key terms.'), (0.8203, 'A well-tuned term extractor improves the precision of a search engine.'), (0.0, 'Indexing relies heavily on how terms are weighted across a corpus.'), (0.0, 'Term weighting strategies often depend on sparse vector representations.')],1):
    print(f"  {i}. {s:.4f}  {d}")
print()
print("Stage 2b -- CrossEncoder (ms-marco-MiniLM-L-6-v2)")
for i,(s,d) in enumerate([(0.853, 'A well-tuned term extractor improves the precision of a search engine.'), (-7.4414, 'Statistical models help automate the process of identifying key terms.'), (-10.7988, 'Term weighting strategies often depend on sparse vector representations.'), (-10.9948, 'Indexing relies heavily on how terms are weighted across a corpus.')],1):
    print(f"  {i}. {s:+.4f}  {d}")
print()
print("Stage 2c -- Cohere rerank-v3.5 (API key required)")
print("  Skipped in this run -- see Stage 2c cell above for the live call")

  QUERY: Modern language models boost the precision of automated term extraction.

Stage 1 -- Dense (all-MiniLM-L6-v2)
  1. 0.7084  A well-tuned term extractor improves the precision of a search engine.
  2. 0.5874  Statistical models help automate the process of identifying key terms.
  3. 0.4880  Indexing relies heavily on how terms are weighted across a corpus.
  4. 0.4820  Term weighting strategies often depend on sparse vector representations.

Stage 2a -- rank_bm25 (classic)
  1. 0.8567  Statistical models help automate the process of identifying key terms.
  2. 0.8203  A well-tuned term extractor improves the precision of a search engine.
  3. 0.0000  Indexing relies heavily on how terms are weighted across a corpus.
  4. 0.0000  Term weighting strategies often depend on sparse vector representations.

Stage 2b -- CrossEncoder (ms-marco-MiniLM-L-6-v2)
  1. +0.8530  A well-tuned term extractor improves the precision of a search engine.
  2. -7.4414  Statistical models help automa

In [27]:
print("\nCONSENSUS TOP-2 (agree across BM25 and Cross-Encoder):")
print("  1. 'A well-tuned term extractor improves the precision of a search engine.'")
print("  2. 'Statistical models help automate the process of identifying key terms.'")
print()
print("Strategy: pass only these consensus passages to the LLM -- fewer tokens, lower cost, less hallucination")


CONSENSUS TOP-2 (agree across BM25 and Cross-Encoder):
  1. 'A well-tuned term extractor improves the precision of a search engine.'
  2. 'Statistical models help automate the process of identifying key terms.'

Strategy: pass only these consensus passages to the LLM -- fewer tokens, lower cost, less hallucination


## Section — Decision Guide

```
Commercial product?
├─ Yes → Apache 2.0 → mxbai-rerank-v2 / bge-reranker-v2-m3 / Qwen3-Reranker
└─ No  → jina-reranker-v3 (SOTA BEIR 61.94) also available

Need multilingual?
├─ Yes → bge-m3 embeddings + bge-reranker-v2-m3 / mxbai-rerank-v2 / Cohere v3.5
└─ No  → all-MiniLM-L6-v2 + ms-marco cross-encoder (English baseline)

Have GPU?
├─ No  → APIs: Cohere v3.5 ($2/1k searches) | Voyage rerank-2.5 ($0.05/1M tokens)
└─ Yes → jina-reranker-v3 (SOTA) | Qwen3-Reranker-8B | mxbai-rerank-large-v2

Long passages (>4k tokens)?
├─ Yes → Cohere v4.0-pro/fast (32k ctx) | jina-reranker-v3 (131k ctx)
└─ No  → any of the above

Latency-critical production?
├─ Yes → Cohere v4.0-fast | Voyage rerank-2.5-lite | Ettin 17M (CPU sub-ms)
└─ No  → Cohere v4.0-pro | jina-v3 | Qwen3-8B
```

## Section — Summary & Key Takeaways

| Topic | Year-ago | 2026 |
|---|---|---|
| Cohere client | `cohere.Client()` | **`cohere.ClientV2()`** |
| Cohere model | `rerank-english-v3.0` | **`rerank-v3.5`** / **`rerank-v4.0-pro/fast`** |
| BM25 | `rank_bm25` | **`bm25s`** (500× faster) |
| Embedding | paraphrase-xlm-r | **`all-MiniLM-L6-v2`** / **`BAAI/bge-m3`** |
| CE inference | `.predict(pairs)` | **`.rank(query, docs)`** (v5 new) |
| LangChain CohereRerank | `langchain.retrievers…` | **`from langchain_cohere import CohereRerank`** |
| SOTA open reranker | ms-marco-MiniLM | **jina-reranker-v3** (61.94 BEIR) / **Qwen3-8B** |

> **GitHub:** The complete annotated notebook is committed to github.com/mayankchugh-learning
> Today's recording used clean code — the GitHub version has full inline documentation.

---
*Made with ❤️ by Mayank Chugh | Please Like, Share & Subscribe on YouTube! | @itaienthusiast*